In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import re

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.layers import Input, Embedding, LSTM, Dense,  Concatenate, Layer
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K

import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

nltk.download("wordnet")
nltk.download("omw-1.4")

In [2]:
df_train = pd.read_json('/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab4/Machine-translation-datasets/small-PhoMT/small-train.json')
df_dev = pd.read_json('/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab4/Machine-translation-datasets/small-PhoMT/small-dev.json')
df_test = pd.read_json('/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab4/Machine-translation-datasets/small-PhoMT/small-test.json')

In [3]:
print(len(df_train), len(df_dev), len(df_test))
df_train.head()

20000 2000 2000


,english,vietnamese
0,It begins with a countdown .,Câu chuyện bắt đầu với buổi lễ đếm ngược .
1,"On August 14th , 1947 , a woman in Bombay goes...","Ngày 14 , tháng 8 , năm 1947 , gần nửa đêm , ở..."
2,"Across India , people hold their breath for th...","Cùng lúc , trên khắp đất Ấn , người ta nín thở..."
3,"And at the stroke of midnight , a squirming in...","Khi đồng hồ điểm thời khắc nửa đêm , một đứa t..."
4,"These events form the foundation of "" Midnight...","Những sự kiện này là nền móng tạo nên "" Những ..."


In [4]:
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,¿])", r" \1 ", sentence)
    sentence = re.sub(r"[^a-zA-ZÀ-ỹ?.!,¿]+", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence

In [5]:
df_train["english"] = df_train["english"].apply(preprocess_sentence)
df_train["vietnamese"] = df_train["vietnamese"].apply(preprocess_sentence)

df_test["english"] = df_test["english"].apply(preprocess_sentence)
df_test["vietnamese"] = df_test["vietnamese"].apply(preprocess_sentence)

df_dev["english"] = df_dev["english"].apply(preprocess_sentence)
df_dev["vietnamese"] = df_dev["vietnamese"].apply(preprocess_sentence)

In [6]:
df_train["vietnamese"] = df_train["vietnamese"].apply(
    lambda x: "<start> " + x + " <end>"
)

df_test["vietnamese"] = df_test["vietnamese"].apply(
    lambda x: "<start> " + x + " <end>"
)

df_dev["vietnamese"] = df_dev["vietnamese"].apply(
    lambda x: "<start> " + x + " <end>"
)

In [7]:
SRC_VOCAB_SIZE = 40000
TGT_VOCAB_SIZE = 40000

In [8]:
src_tokenizer = Tokenizer(
    num_words=SRC_VOCAB_SIZE,
    oov_token="<unk>",
    filters=""
)

src_tokenizer.fit_on_texts(df_train["english"])

In [9]:
tgt_tokenizer = Tokenizer(
    num_words=TGT_VOCAB_SIZE,
    oov_token="<unk>",
    filters=""
)

tgt_tokenizer.fit_on_texts(df_train["vietnamese"])


In [10]:
encoder_train_seq = src_tokenizer.texts_to_sequences(df_train["english"])
decoder_train_seq = tgt_tokenizer.texts_to_sequences(df_train["vietnamese"])

encoder_dev_seq = src_tokenizer.texts_to_sequences(df_dev["english"])
decoder_dev_seq = tgt_tokenizer.texts_to_sequences(df_dev["vietnamese"])


In [ ]:
MAX_LEN_SRC = int(np.percentile([len(x) for x in encoder_train_seq], 95))
MAX_LEN_TGT = int(np.percentile([len(x) for x in decoder_train_seq], 95))


In [12]:
encoder_train = pad_sequences(
    encoder_train_seq, maxlen=MAX_LEN_SRC, padding="post"
)

decoder_train = pad_sequences(
    decoder_train_seq, maxlen=MAX_LEN_TGT, padding="post"
)

encoder_dev = pad_sequences(
    encoder_dev_seq, maxlen=MAX_LEN_SRC, padding="post"
)

decoder_dev = pad_sequences(
    decoder_dev_seq, maxlen=MAX_LEN_TGT, padding="post"
)

In [13]:
decoder_input_train = decoder_train[:, :-1]
decoder_output_train = decoder_train[:, 1:]

decoder_input_dev = decoder_dev[:, :-1]
decoder_output_dev = decoder_dev[:, 1:]

In [14]:
print("Encoder input:", encoder_train.shape)
print("Decoder input:", decoder_input_train.shape)
print("Decoder output:", decoder_output_train.shape)

Encoder input: (20000, 45)
Decoder input: (20000, 55)
Decoder output: (20000, 55)


In [15]:
EMBED_DIM = 256
HIDDEN_SIZE = 256
NUM_LAYERS = 5

SRC_VOCAB_SIZE = min(SRC_VOCAB_SIZE, len(src_tokenizer.word_index) + 1)
TGT_VOCAB_SIZE = min(TGT_VOCAB_SIZE, len(tgt_tokenizer.word_index) + 1)

# Bài 1:

In [ ]:
class Seq2SeqLSTM:
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        max_len_src,
        max_len_tgt,
        embed_dim=256,
        hidden_size=256,
        num_layers=5,
        learning_rate=1e-3
    ):
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.max_len_src = max_len_src
        self.max_len_tgt = max_len_tgt
        self.embed_dim = embed_dim
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.learning_rate = learning_rate

        self._build_model()
    def _build_model(self):
        # ===== Encoder =====
        encoder_inputs = Input(
            shape=(self.max_len_src,), name="encoder_inputs"
        )

        encoder_embedding = Embedding(
            self.src_vocab_size,
            self.embed_dim,
            mask_zero=True,
            name="encoder_embedding"
        )(encoder_inputs)

        x = encoder_embedding
        for i in range(self.num_layers):
            lstm = LSTM(
                self.hidden_size,
                return_sequences=(i < self.num_layers - 1),
                return_state=True,
                name=f"encoder_lstm_{i+1}"
            )
            if i < self.num_layers - 1:
                x, _, _ = lstm(x)
            else:
                _, state_h, state_c = lstm(x)

        encoder_states = [state_h, state_c]

        # ===== Decoder =====
        decoder_inputs = Input(
            shape=(self.max_len_tgt - 1,), name="decoder_inputs"
        )

        decoder_embedding = Embedding(
            self.tgt_vocab_size,
            self.embed_dim,
            mask_zero=True,
            name="decoder_embedding"
        )(decoder_inputs)

        x = decoder_embedding

        # LSTM layer 1 nhận state từ encoder
        decoder_lstm_1 = LSTM(
            self.hidden_size,
            return_sequences=True,
            return_state=True,
            name="decoder_lstm_1"
        )
        x, _, _ = decoder_lstm_1(
            x, initial_state=encoder_states
        )

        # Các LSTM decoder còn lại
        for i in range(1, self.num_layers):
            lstm = LSTM(
                self.hidden_size,
                return_sequences=True,
                return_state=True,
                name=f"decoder_lstm_{i+1}"
            )
            x, _, _ = lstm(x)

        decoder_outputs = Dense(
            self.tgt_vocab_size,
            activation="softmax",
            name="decoder_output"
        )(x)

        # ===== Model =====
        self.model = Model(
            inputs=[encoder_inputs, decoder_inputs],
            outputs=decoder_outputs
        )

        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=self.learning_rate
            ),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
    def summary(self):
        self.model.summary()

    def train(
        self,
        encoder_input,
        decoder_input,
        decoder_output,
        encoder_val,
        decoder_val,
        decoder_output_val,
        batch_size=64,
        epochs=20
    ):
        return self.model.fit(
            [encoder_input, decoder_input],
            decoder_output,
            validation_data=(
                [encoder_val, decoder_val],
                decoder_output_val
            ),
            batch_size=batch_size,
            epochs=epochs
        )


In [18]:
seq2seq = Seq2SeqLSTM(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    max_len_src=MAX_LEN_SRC,
    max_len_tgt=MAX_LEN_TGT,
    embed_dim=256,
    hidden_size=256,
    num_layers=5
)

seq2seq.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 45)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 45, 256)   │  4,282,112 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 45)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_1      │ [(None, 45, 256), │    525,312 │ encoder_embeddin… │
│ (LSTM)              │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_2      │ [(None, 45, 256), │    525,312 │ encoder_lstm_1[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_3      │ [(None, 45, 256), │    525,312 │ encoder_lstm_2[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 55)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_4      │ [(None, 45, 256), │    525,312 │ encoder_lstm_3[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 55, 256)   │  1,660,416 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_5      │ [(None, 256),     │    525,312 │ encoder_lstm_4[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm_1      │ [(None, 55, 256), │    525,312 │ decoder_embeddin… │
│ (LSTM)              │ (None, 256),      │            │ encoder_lstm_5[0… │
│                     │ (None, 256)]      │            │ encoder_lstm_5[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 55)        │          0 │ decoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm_2      │ [(None, 55, 256), │    525,312 │ decoder_lstm_1[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_1[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm_3      │ [(None, 55, 256), │    525,312 │ decoder_lstm_2[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_1[0][0] │
│                     │ (None, 256)]      │            │                 

 Total params: 12,862,550 (49.07 MB)

 Trainable params: 12,862,550 (49.07 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
history = seq2seq.train(
    encoder_train,
    decoder_input_train,
    decoder_output_train[..., None],
    encoder_dev,
    decoder_input_dev,
    decoder_output_dev[..., None],
    batch_size=64,
    epochs=10
)


Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 43s 97ms/step - accuracy: 0.0388 - loss: 6.5407 - val_accuracy: 0.0231 - val_loss: 6.3616
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 95ms/step - accuracy: 0.0232 - loss: 6.0866 - val_accuracy: 0.0250 - val_loss: 6.3568
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 95ms/step - accuracy: 0.0281 - loss: 6.0305 - val_accuracy: 0.0439 - val_loss: 6.1682
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 96ms/step - accuracy: 0.0471 - loss: 5.8233 - val_accuracy: 0.0694 - val_loss: 6.0170
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 95ms/step - accuracy: 0.0665 - loss: 5.6587 - val_accuracy: 0.0699 - val_loss: 5.9082
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 95ms/step - accuracy: 0.0681 - loss: 5.5518 - val_accuracy: 0.0758 - val_loss: 5.7688
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 95ms/step - accuracy: 0.0739 - loss: 5.3879 - val_accuracy: 0.0791 - val_loss: 5.6241
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 96ms/step - accuracy: 0.0792 - loss: 5.1971 - 

In [41]:
def build_encoder_inference(model):
    encoder_inputs = model.inputs[0]   # ✅ FIX
    encoder_lstm = model.get_layer("encoder_lstm_5")

    _, state_h, state_c = encoder_lstm.output

    encoder_model = tf.keras.Model(
        inputs=encoder_inputs,
        outputs=[state_h, state_c]
    )
    return encoder_model

In [42]:
def build_decoder_inference(model, hidden_size):
    decoder_inputs = model.inputs[1]   # ✅ FIX luôn cho nhất quán
    decoder_embedding = model.get_layer("decoder_embedding")(decoder_inputs)

    state_input_h = tf.keras.Input(shape=(hidden_size,))
    state_input_c = tf.keras.Input(shape=(hidden_size,))
    states_inputs = [state_input_h, state_input_c]

    x = decoder_embedding

    lstm1 = model.get_layer("decoder_lstm_1")
    x, h, c = lstm1(x, initial_state=states_inputs)

    for i in range(2, 6):
        lstm = model.get_layer(f"decoder_lstm_{i}")
        x, h, c = lstm(x)

    dense = model.get_layer("decoder_output")
    outputs = dense(x)

    decoder_model = tf.keras.Model(
        [decoder_inputs] + states_inputs,
        [outputs, h, c]
    )
    return decoder_model


In [43]:
encoder_model = build_encoder_inference(seq2seq.model)
decoder_model = build_decoder_inference(seq2seq.model, hidden_size=256)

In [44]:
def encode_batch(sentences, tokenizer, max_len):
    seqs = tokenizer.texts_to_sequences(sentences)
    return pad_sequences(seqs, maxlen=max_len, padding="post")

In [45]:
def batch_translate(
    sentences,
    encoder_model,
    decoder_model,
    src_tokenizer,
    tgt_tokenizer,
    max_len_src,
    max_len_tgt,
    batch_size=64
):
    start_id = tgt_tokenizer.word_index["<start>"]
    end_id = tgt_tokenizer.word_index["<end>"]

    all_predictions = []

    for i in tqdm(
        range(0, len(sentences), batch_size),
        desc="Batch decoding"
    ):
        batch_sents = sentences[i:i + batch_size]
        bs = len(batch_sents)

        # ---- Encoder: 1 lần cho cả batch ----
        enc_input = encode_batch(
            batch_sents, src_tokenizer, max_len_src
        )
        state_h, state_c = encoder_model.predict(enc_input, verbose=0)

        # ---- Decoder init ----
        cur_tokens = np.full((bs, 1), start_id, dtype=np.int32)
        finished = np.zeros(bs, dtype=bool)
        decoded = [[] for _ in range(bs)]

        # ---- Decode theo timestep (vectorized) ----
        for _ in range(max_len_tgt):
            outputs, state_h, state_c = decoder_model.predict(
                [cur_tokens, state_h, state_c],
                verbose=0
            )

            next_tokens = outputs[:, -1, :].argmax(axis=-1)

            for j, tok in enumerate(next_tokens):
                if not finished[j]:
                    if tok == end_id:
                        finished[j] = True
                    else:
                        decoded[j].append(
                            tgt_tokenizer.index_word.get(tok, "<unk>")
                        )

            if finished.all():
                break

            cur_tokens = next_tokens.reshape(-1, 1)

        all_predictions.extend(decoded)

    return all_predictions


In [46]:
predictions = batch_translate(
    df_test["english"].tolist(),
    encoder_model,
    decoder_model,
    src_tokenizer,
    tgt_tokenizer,
    MAX_LEN_SRC,
    MAX_LEN_TGT,
    batch_size=64
)

Batch decoding: 100%|██████████| 32/32 [02:38<00:00,  4.95s/it]


In [47]:
references = [
    [vi.replace("<start>", "").replace("<end>", "").strip().split()]
    for vi in df_test["vietnamese"]
]

In [48]:
smooth = SmoothingFunction().method4

bleu_1 = corpus_bleu(references, predictions, weights=(1, 0, 0, 0), smoothing_function=smooth)
bleu_2 = corpus_bleu(references, predictions, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
bleu_3 = corpus_bleu(references, predictions, weights=(1/3, 1/3, 1/3, 0), smoothing_function=smooth)
bleu_4 = corpus_bleu(references, predictions, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)

print(f"BLEU-1: {bleu_1:.4f}")
print(f"BLEU-2: {bleu_2:.4f}")
print(f"BLEU-3: {bleu_3:.4f}")
print(f"BLEU-4: {bleu_4:.4f}")


BLEU-1: 0.0118
BLEU-2: 0.0019
BLEU-3: 0.0003
BLEU-4: 0.0001


In [51]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge1, rouge2, rougel, meteor = [], [], [], []

for ref, pred in tqdm(
    zip(references, predictions),
    total=len(predictions),
    desc="Scoring"
):
    ref_text = " ".join(ref[0])
    pred_text = " ".join(pred)

    r = scorer.score(ref_text, pred_text)
    rouge1.append(r["rouge1"].fmeasure)
    rouge2.append(r["rouge2"].fmeasure)
    rougel.append(r["rougeL"].fmeasure)

    meteor.append(meteor_score(ref, pred))

print(f"ROUGE-1: {np.mean(rouge1):.4f}")
print(f"ROUGE-2: {np.mean(rouge2):.4f}")
print(f"ROUGE-L: {np.mean(rougel):.4f}")
print(f"METEOR : {np.mean(meteor):.4f}")

Scoring: 100%|██████████| 2000/2000 [00:08<00:00, 231.68it/s]

ROUGE-1: 0.0787
ROUGE-2: 0.0102
ROUGE-L: 0.0740
METEOR : 0.0111


# Bài 2:

In [17]:
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, hidden_size):
        super().__init__()
        self.W_q = Dense(hidden_size, use_bias=False)
        self.W_k = Dense(hidden_size, use_bias=False)
        self.v = Dense(1, use_bias=False)

    def call(self, decoder_outputs, encoder_outputs):
        query = tf.expand_dims(self.W_q(decoder_outputs), 2)

        keys = tf.expand_dims(self.W_k(encoder_outputs), 1)

        energy = tf.nn.tanh(query + keys)

        scores = tf.squeeze(self.v(energy), axis=-1)

        attention_weights = tf.nn.softmax(scores, axis=-1)

        context = tf.matmul(attention_weights, encoder_outputs)

        return context

In [18]:
class Seq2SeqLSTM:
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        max_len_src,
        max_len_tgt,
        embed_dim=256,
        hidden_size=256,
        num_layers=5,
        learning_rate=1e-3
    ):
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.max_len_src = max_len_src
        self.max_len_tgt = max_len_tgt
        self.embed_dim = embed_dim
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.learning_rate = learning_rate

        self._build_model()

    def _build_model(self):
        encoder_inputs = Input(
            shape=(self.max_len_src,), name="encoder_inputs"
        )

        encoder_embedding = Embedding(
            self.src_vocab_size,
            self.embed_dim,
            mask_zero=True,
            name="encoder_embedding"
        )(encoder_inputs)

        x = encoder_embedding

        for i in range(self.num_layers):
            lstm = LSTM(
                self.hidden_size,
                return_sequences=True,
                return_state=True,
                name=f"encoder_lstm_{i+1}"
            )
            x, state_h, state_c = lstm(x)

        encoder_outputs = x
        encoder_states = [state_h, state_c]

        decoder_inputs = Input(
            shape=(self.max_len_tgt - 1,), name="decoder_inputs"
        )

        decoder_embedding = Embedding(
            self.tgt_vocab_size,
            self.embed_dim,
            mask_zero=True,
            name="decoder_embedding"
        )(decoder_inputs)

        x = decoder_embedding

        decoder_lstm_1 = LSTM(
            self.hidden_size,
            return_sequences=True,
            return_state=True,
            name="decoder_lstm_1"
        )
        x, _, _ = decoder_lstm_1(
            x, initial_state=encoder_states
        )

        for i in range(1, self.num_layers):
            lstm = LSTM(
                self.hidden_size,
                return_sequences=True,
                return_state=True,
                name=f"decoder_lstm_{i+1}"
            )
            x, _, _ = lstm(x)

        decoder_outputs = x

        attention = BahdanauAttention(self.hidden_size)
        context = attention(decoder_outputs, encoder_outputs)

        concat = Concatenate(axis=-1)(
            [decoder_outputs, context]
        )

        outputs = Dense(
            self.tgt_vocab_size,
            activation="softmax",
            name="decoder_output"
        )(concat)

        self.model = Model(
            inputs=[encoder_inputs, decoder_inputs],
            outputs=outputs
        )

        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=self.learning_rate
            ),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

    def summary(self):
        self.model.summary()

    def train(
        self,
        train_ds,
        val_ds,
        epochs=10
    ):
        return self.model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs
        )

In [19]:
BATCH_SIZE = 64
BUFFER_SIZE = 20000

In [20]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
            "encoder_inputs": encoder_train,
            "decoder_inputs": decoder_input_train
        },
        decoder_output_train
    )
)

train_ds = (
    train_ds
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [21]:
val_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
            "encoder_inputs": encoder_dev,
            "decoder_inputs": decoder_input_dev
        },
        decoder_output_dev
    )
)

val_ds = (
    val_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [22]:
for batch in train_ds.take(1):
    x, y = batch
    print("Encoder input shape:", x["encoder_inputs"].shape)
    print("Decoder input shape:", x["decoder_inputs"].shape)
    print("Decoder output shape:", y.shape)

Encoder input shape: (64, 45)
Decoder input shape: (64, 55)
Decoder output shape: (64, 55)


In [23]:
model = Seq2SeqLSTM(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    max_len_src=MAX_LEN_SRC,
    max_len_tgt=MAX_LEN_TGT,
    embed_dim=256,
    hidden_size=256,
    num_layers=5
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'bahdanau_attention' (of type BahdanauAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 45)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 45, 256)   │  4,282,112 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 45)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_1      │ [(None, 45, 256), │    525,312 │ encoder_embeddin… │
│ (LSTM)              │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_2      │ [(None, 45, 256), │    525,312 │ encoder_lstm_1[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_3      │ [(None, 45, 256), │    525,312 │ encoder_lstm_2[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 55)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_4      │ [(None, 45, 256), │    525,312 │ encoder_lstm_3[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 55, 256)   │  1,660,416 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_5      │ [(None, 45, 256), │    525,312 │ encoder_lstm_4[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm_1      │ [(None, 55, 256), │    525,312 │ decoder_embeddin… │
│ (LSTM)              │ (None, 256),      │            │ encoder_lstm_5[0… │
│                     │ (None, 256)]      │            │ encoder_lstm_5[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 55)        │          0 │ decoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm_2      │ [(None, 55, 256), │    525,312 │ decoder_lstm_1[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_1[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm_3      │ [(None, 55, 256), │    525,312 │ decoder_lstm_2[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_1[0][0] │
│                     │ (None, 256)]      │            │                 

 Total params: 14,654,294 (55.90 MB)

 Trainable params: 14,654,294 (55.90 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
history = model.train(
    train_ds=train_ds,
    val_ds=val_ds,
    epochs=10
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 117ms/step - accuracy: 0.5662 - loss: 3.7473 - val_accuracy: 0.5919 - val_loss: 2.8657
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 115ms/step - accuracy: 0.6197 - loss: 2.5252 - val_accuracy: 0.5972 - val_loss: 2.7664
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 114ms/step - accuracy: 0.6288 - loss: 2.3918 - val_accuracy: 0.5997 - val_loss: 2.7099
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 114ms/step - accuracy: 0.6286 - loss: 2.3504 - val_accuracy: 0.6019 - val_loss: 2.6772
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 114ms/step - accuracy: 0.6334 - loss: 2.2951 - val_accuracy: 0.6040 - val_loss: 2.6433
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 114ms/step - accuracy: 0.6363 - loss: 2.2486 - val_accuracy: 0.6071 - val_loss: 2.6086
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 115ms/step - accuracy: 0.6417 - loss: 2.1969 - val_accuracy: 0.6112 - val_loss: 2.5702
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 114ms/step - accuracy: 0.6448 - loss: 2

In [44]:
START_TOKEN = tgt_tokenizer.word_index["<start>"]
END_TOKEN   = tgt_tokenizer.word_index["<end>"]

index_to_word = {v: k for k, v in tgt_tokenizer.word_index.items()}

In [ ]:
encoder_test_seq = src_tokenizer.texts_to_sequences(df_test["english"])
encoder_test = pad_sequences(
    encoder_test_seq,
    maxlen=MAX_LEN_SRC,
    padding="post"
)

decoder_test_seq = tgt_tokenizer.texts_to_sequences(df_test["vietnamese"])
decoder_test = pad_sequences(
    decoder_test_seq,
    maxlen=MAX_LEN_TGT,
    padding="post"
)

decoder_input_test = decoder_test[:, :-1]
decoder_output_test = decoder_test[:, 1:]

In [61]:
def decode_sentence(token_ids):
    words = []
    for t in token_ids:
        if t == 0:
            continue
        w = index_to_word.get(int(t), "<unk>")
        if w not in ["<start>", "<end>"]:
            words.append(w)
    return " ".join(words)

In [ ]:
def predict_test_set(
    model,
    encoder_test,
    decoder_input_test,
    decoder_output_test,
    batch_size=64
):
    logits = model.predict(
        [encoder_test, decoder_input_test],
        batch_size=batch_size,
        verbose=1
    )

    pred_ids = np.argmax(logits, axis=-1)

    predictions = []
    references = []

    for i in range(len(pred_ids)):
        pred_sentence = decode_sentence(pred_ids[i])
        ref_sentence  = decode_sentence(decoder_output_test[i])

        predictions.append(pred_sentence.split())
        references.append([ref_sentence.split()])

    return predictions, references


In [63]:
preds, refs = predict_test_set(
    model.model,
    encoder_test,
    decoder_input_test,
    decoder_output_test,
    batch_size=64
)

32/32 ━━━━━━━━━━━━━━━━━━━━ 7s 139ms/step


In [ ]:
smooth = SmoothingFunction().method4

bleu_1 = corpus_bleu(
    refs, preds,
    weights=(1, 0, 0, 0),
    smoothing_function=smooth
)

bleu_2 = corpus_bleu(
    refs, preds,
    weights=(0.5, 0.5, 0, 0),
    smoothing_function=smooth
)

bleu_3 = corpus_bleu(
    refs, preds,
    weights=(1/3, 1/3, 1/3, 0),
    smoothing_function=smooth
)

bleu_4 = corpus_bleu(
    refs, preds,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smooth
)

print(f"BLEU@1: {bleu_1:.4f}")
print(f"BLEU@2: {bleu_2:.4f}")
print(f"BLEU@3: {bleu_3:.4f}")
print(f"BLEU@4: {bleu_4:.4f}")

BLEU@1: 0.2065
BLEU@2: 0.0715
BLEU@3: 0.0269
BLEU@4: 0.0110


In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge_1, rouge_2, rouge_L = [], [], []

for pred, ref in zip(preds, refs):
    pred_text = " ".join(pred)
    ref_text  = " ".join(ref[0])

    scores = scorer.score(ref_text, pred_text)

    rouge_1.append(scores["rouge1"].fmeasure)
    rouge_2.append(scores["rouge2"].fmeasure)
    rouge_L.append(scores["rougeL"].fmeasure)

print(f"ROUGE-1 F1: {np.mean(rouge_1):.4f}")
print(f"ROUGE-2 F1: {np.mean(rouge_2):.4f}")
print(f"ROUGE-L F1: {np.mean(rouge_L):.4f}")

ROUGE-1 F1: 0.4701
ROUGE-2 F1: 0.1180
ROUGE-L F1: 0.3191


In [ ]:
meteor_scores = []

for pred, ref in zip(preds, refs):
    score = meteor_score(
        ref,
        pred
    )
    meteor_scores.append(score)

print(f"METEOR: {np.mean(meteor_scores):.4f}")


METEOR: 0.1199


# Bài 3:

In [ ]:
class LuongAttention(Layer):
    """
    Luong Multiplicative Attention
    score(s_t, h_i) = s_t^T W h_i
    """
    def __init__(self, hidden_size, **kwargs):
        super().__init__(**kwargs)
        self.hidden_size = hidden_size
        self.W = Dense(hidden_size, use_bias=False)

    def call(self, decoder_outputs, encoder_outputs):
        """
        decoder_outputs: (batch, tgt_len, hidden)
        encoder_outputs: (batch, src_len, hidden)
        """

        # W * h_i
        keys = self.W(encoder_outputs)  # (batch, src_len, hidden)

        # score = s_t^T * W * h_i
        scores = tf.matmul(
            decoder_outputs,
            keys,
            transpose_b=True
        )  # (batch, tgt_len, src_len)

        attention_weights = tf.nn.softmax(scores, axis=-1)

        # context vector
        context = tf.matmul(
            attention_weights,
            encoder_outputs
        )  # (batch, tgt_len, hidden)

        return context

In [ ]:
class Seq2SeqLSTM_Luong:
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        max_len_src,
        max_len_tgt,
        embed_dim=256,
        hidden_size=256,
        num_layers=5,
        learning_rate=1e-3
    ):
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.max_len_src = max_len_src
        self.max_len_tgt = max_len_tgt
        self.embed_dim = embed_dim
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.learning_rate = learning_rate

        self._build_model()

    def _build_model(self):
        # ================= ENCODER =================
        encoder_inputs = Input(
            shape=(self.max_len_src,),
            name="encoder_inputs"
        )

        encoder_embed = Embedding(
            self.src_vocab_size,
            self.embed_dim,
            mask_zero=True,
            name="encoder_embedding"
        )(encoder_inputs)

        x = encoder_embed
        for i in range(self.num_layers):
            lstm = LSTM(
                self.hidden_size,
                return_sequences=True,
                return_state=True,
                name=f"encoder_lstm_{i+1}"
            )
            x, h, c = lstm(x)

        encoder_outputs = x
        encoder_states = [h, c]

        # ================= DECODER =================
        decoder_inputs = Input(
            shape=(self.max_len_tgt - 1,),
            name="decoder_inputs"
        )

        decoder_embed = Embedding(
            self.tgt_vocab_size,
            self.embed_dim,
            mask_zero=True,
            name="decoder_embedding"
        )(decoder_inputs)

        x = decoder_embed

        # Decoder LSTM 1 nhận state từ encoder
        lstm1 = LSTM(
            self.hidden_size,
            return_sequences=True,
            return_state=True,
            name="decoder_lstm_1"
        )
        x, _, _ = lstm1(x, initial_state=encoder_states)

        # Các layer còn lại
        for i in range(1, self.num_layers):
            lstm = LSTM(
                self.hidden_size,
                return_sequences=True,
                return_state=True,
                name=f"decoder_lstm_{i+1}"
            )
            x, _, _ = lstm(x)

        decoder_outputs = x  # (batch, tgt_len, hidden)

        # ================= LUONG ATTENTION =================
        attention = LuongAttention(
            self.hidden_size,
            name="luong_attention"
        )

        context = attention(
            decoder_outputs,
            encoder_outputs
        )

        # Kết hợp context + decoder output
        combined = Concatenate(axis=-1)(
            [decoder_outputs, context]
        )

        # Output layer
        outputs = Dense(
            self.tgt_vocab_size,
            activation="softmax",
            name="decoder_output"
        )(combined)

        # ================= MODEL =================
        self.model = Model(
            inputs=[encoder_inputs, decoder_inputs],
            outputs=outputs
        )

        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=self.learning_rate
            ),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

    def summary(self):
        self.model.summary()


In [71]:
seq2seq_luong = Seq2SeqLSTM_Luong(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    max_len_src=MAX_LEN_SRC,
    max_len_tgt=MAX_LEN_TGT,
    embed_dim=256,
    hidden_size=256,
    num_layers=5,
    learning_rate=1e-3
)

seq2seq_luong.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'luong_attention' (of type LuongAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 45)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 45, 256)   │  4,282,112 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 45)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_1      │ [(None, 45, 256), │    525,312 │ encoder_embeddin… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_2[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_2      │ [(None, 45, 256), │    525,312 │ encoder_lstm_1[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_2[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_3      │ [(None, 45, 256), │    525,312 │ encoder_lstm_2[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_2[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 55)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_4      │ [(None, 45, 256), │    525,312 │ encoder_lstm_3[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_2[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 55, 256)   │  1,660,416 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_5      │ [(None, 45, 256), │    525,312 │ encoder_lstm_4[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_2[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm_1      │ [(None, 55, 256), │    525,312 │ decoder_embeddin… │
│ (LSTM)              │ (None, 256),      │            │ encoder_lstm_5[0… │
│                     │ (None, 256)]      │            │ encoder_lstm_5[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 55)        │          0 │ decoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm_2      │ [(None, 55, 256), │    525,312 │ decoder_lstm_1[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_3[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm_3      │ [(None, 55, 256), │    525,312 │ decoder_lstm_2[0… │
│ (LSTM)              │ (None, 256),      │            │ not_equal_3[0][0] │
│                     │ (None, 256)]      │            │                 

 Total params: 14,588,502 (55.65 MB)

 Trainable params: 14,588,502 (55.65 MB)

 Non-trainable params: 0 (0.00 B)

In [72]:
history = seq2seq_luong.model.fit(
    [encoder_train, decoder_input_train],
    decoder_output_train,
    validation_data=(
        [encoder_dev, decoder_input_dev],
        decoder_output_dev
    ),
    batch_size=64,
    epochs=20
)


Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 43s 112ms/step - accuracy: 0.5650 - loss: 5.2366 - val_accuracy: 0.5586 - val_loss: 3.4016
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 114ms/step - accuracy: 0.6109 - loss: 2.9182 - val_accuracy: 0.5942 - val_loss: 3.1143
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 106ms/step - accuracy: 0.6233 - loss: 2.6197 - val_accuracy: 0.5963 - val_loss: 3.0421
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 105ms/step - accuracy: 0.6309 - loss: 2.4878 - val_accuracy: 0.5969 - val_loss: 2.9978
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 105ms/step - accuracy: 0.6314 - loss: 2.4261 - val_accuracy: 0.6051 - val_loss: 2.9436
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 105ms/step - accuracy: 0.6354 - loss: 2.3609 - val_accuracy: 0.6069 - val_loss: 2.9292
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 104ms/step - accuracy: 0.6396 - loss: 2.3041 - val_accuracy: 0.6082 - val_loss: 2.8985
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 105ms/step - accuracy: 0.6412 - loss: 2

In [73]:
preds, refs = predict_test_set(
    seq2seq_luong.model,
    encoder_test,
    decoder_input_test,
    decoder_output_test,
    batch_size=64
)

32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 116ms/step


In [ ]:
smooth = SmoothingFunction().method4

bleu_1 = corpus_bleu(
    refs, preds,
    weights=(1, 0, 0, 0),
    smoothing_function=smooth
)

bleu_2 = corpus_bleu(
    refs, preds,
    weights=(0.5, 0.5, 0, 0),
    smoothing_function=smooth
)

bleu_3 = corpus_bleu(
    refs, preds,
    weights=(1/3, 1/3, 1/3, 0),
    smoothing_function=smooth
)

bleu_4 = corpus_bleu(
    refs, preds,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smooth
)

print(f"BLEU@1: {bleu_1:.4f}")
print(f"BLEU@2: {bleu_2:.4f}")
print(f"BLEU@3: {bleu_3:.4f}")
print(f"BLEU@4: {bleu_4:.4f}")

BLEU@1: 0.2382
BLEU@2: 0.0795
BLEU@3: 0.0255
BLEU@4: 0.0090


In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge_1, rouge_2, rouge_L = [], [], []

for pred, ref in zip(preds, refs):
    pred_text = " ".join(pred)
    ref_text  = " ".join(ref[0])

    scores = scorer.score(ref_text, pred_text)

    rouge_1.append(scores["rouge1"].fmeasure)
    rouge_2.append(scores["rouge2"].fmeasure)
    rouge_L.append(scores["rougeL"].fmeasure)

print(f"ROUGE-1 F1: {np.mean(rouge_1):.4f}")
print(f"ROUGE-2 F1: {np.mean(rouge_2):.4f}")
print(f"ROUGE-L F1: {np.mean(rouge_L):.4f}")

ROUGE-1 F1: 0.4881
ROUGE-2 F1: 0.1179
ROUGE-L F1: 0.3149


In [ ]:
meteor_scores = []

for pred, ref in zip(preds, refs):
    score = meteor_score(
        ref,
        pred
    )
    meteor_scores.append(score)

print(f"METEOR: {np.mean(meteor_scores):.4f}")

METEOR: 0.1328
